# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

rel = "hf://datasets/FlyRank/internship-warehouse"

Token loaded successfully!
Connected successfully!


In [3]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
"""

df = con.sql(query).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


In [4]:
feature_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE NULL
    END AS ctr,

    CASE
        WHEN ga4_sessions > 0
        THEN ga4_engaged_sessions * 1.0 / ga4_sessions
        ELSE NULL
    END AS engagement_rate,

    LN(1 + gsc_impressions) AS log_impressions,
    LN(1 + gsc_clicks) AS log_clicks,
    LN(1 + ga4_sessions) AS log_sessions,
    LN(1 + ga4_engaged_sessions) AS log_engaged_sessions

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
LIMIT 100000
"""

feature_df = con.sql(feature_query).df()

print("Rows:", len(feature_df))
print("Columns:")
print(feature_df.columns.tolist())

feature_df.head()

Rows: 100000
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'log_impressions', 'log_clicks', 'log_sessions', 'log_engaged_sessions']


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate,log_impressions,log_clicks,log_sessions,log_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,0.000,NaN,3.044522,0.000000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,0.000,NaN,0.693147,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,0.008,NaN,4.836282,0.693147,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,0.000,NaN,2.079442,0.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,0.000,NaN,2.484907,0.000000,NaN,NaN


In [5]:
missing_summary = feature_df.isna().sum()

print("Missing values per feature:")
print(missing_summary)

print("\nMissing percentage:")
print((feature_df.isna().mean() * 100).round(2))

Missing values per feature:
report_date                 0
client_hash_id              0
content_hash_id             0
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position        60249
ga4_sessions            26561
ga4_engaged_sessions    26561
ctr                     60249
engagement_rate         96097
log_impressions             0
log_clicks                  0
log_sessions            26561
log_engaged_sessions    26561
dtype: int64

Missing percentage:
report_date              0.00
client_hash_id           0.00
content_hash_id          0.00
gsc_impressions          0.00
gsc_clicks               0.00
gsc_avg_position        60.25
ga4_sessions            26.56
ga4_engaged_sessions    26.56
ctr                     60.25
engagement_rate         96.10
log_impressions          0.00
log_clicks               0.00
log_sessions            26.56
log_engaged_sessions    26.56
dtype: float64


In [6]:
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN gsc_data_available = TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_unavailable,

    SUM(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END) AS ga4_available,
    SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS ga4_unavailable

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
"""

availability_df = con.sql(availability_query).df()

availability_df

,total_rows,gsc_available,gsc_unavailable,ga4_available,ga4_unavailable
0,9841378,3611061.0,6230317.0,413966.0,6408671.0


In [7]:
feature_query = f"""
SELECT
    report_date,

    -- Core numeric features
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,

    -- Availability indicators
    gsc_data_available,
    ga4_data_available,

    -- Engineered features
    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE NULL
    END AS ctr,

    CASE
        WHEN ga4_data_available = TRUE
             AND ga4_sessions > 0
        THEN ga4_engaged_sessions * 1.0 / ga4_sessions
        ELSE NULL
    END AS engagement_rate,

    LN(1 + gsc_impressions) AS log_impressions,
    LN(1 + gsc_clicks) AS log_clicks,

    CASE
        WHEN ga4_data_available = TRUE
        THEN LN(1 + ga4_sessions)
        ELSE NULL
    END AS log_sessions,

    CASE
        WHEN ga4_data_available = TRUE
        THEN LN(1 + ga4_engaged_sessions)
        ELSE NULL
    END AS log_engaged_sessions

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
LIMIT 100000
"""

feature_df = con.sql(feature_query).df()

print("Rows:", len(feature_df))
print("Columns:")
print(feature_df.columns.tolist())

feature_df.head()

Rows: 100000
Columns:
['report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'gsc_data_available', 'ga4_data_available', 'ctr', 'engagement_rate', 'log_impressions', 'log_clicks', 'log_sessions', 'log_engaged_sessions']


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available,ga4_data_available,ctr,engagement_rate,log_impressions,log_clicks,log_sessions,log_engaged_sessions
0,2026-03-01,20,0,3.350000,<NA>,<NA>,True,<NA>,0.000,NaN,3.044522,0.000000,NaN,NaN
1,2026-03-01,1,0,0.000000,<NA>,<NA>,True,<NA>,0.000,NaN,0.693147,0.000000,NaN,NaN
2,2026-03-01,125,1,4.928000,<NA>,<NA>,True,<NA>,0.008,NaN,4.836282,0.693147,NaN,NaN
3,2026-03-01,7,0,4.000000,<NA>,<NA>,True,<NA>,0.000,NaN,2.079442,0.000000,NaN,NaN
4,2026-03-01,11,0,2.272727,<NA>,<NA>,True,<NA>,0.000,NaN,2.484907,0.000000,NaN,NaN


In [8]:
availability_check = feature_df.groupby(
    ["gsc_data_available", "ga4_data_available"],
    dropna=False
).size().reset_index(name="rows")

print(availability_check)

   gsc_data_available  ga4_data_available   rows
0               False               False  51199
1               False                True    314
2               False                <NA>   8736
3                True               False  18333
4                True                True   3593
5                True                <NA>  17825


In [9]:
missing_by_availability = feature_df.groupby(
    ["gsc_data_available", "ga4_data_available"],
    dropna=False
).agg(
    rows=("report_date", "size"),
    missing_position=("gsc_avg_position", lambda x: x.isna().sum()),
    missing_ga4_sessions=("ga4_sessions", lambda x: x.isna().sum()),
    missing_ga4_engaged=("ga4_engaged_sessions", lambda x: x.isna().sum())
).reset_index()

print(missing_by_availability)

   gsc_data_available  ga4_data_available   rows  missing_position  \
0               False               False  51199             51199   
1               False                True    314               314   
2               False                <NA>   8736              8736   
3                True               False  18333                 0   
4                True                True   3593                 0   
5                True                <NA>  17825                 0   

   missing_ga4_sessions  missing_ga4_engaged  
0                     0                    0  
1                     0                    0  
2                  8736                 8736  
3                     0                    0  
4                     0                    0  
5                 17825                17825  


In [10]:
numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate",
    "log_impressions",
    "log_clicks",
    "log_sessions",
    "log_engaged_sessions"
]

print("Median values:")
print(feature_df[numeric_features].median())

Median values:
gsc_impressions              0.0
gsc_clicks                   0.0
gsc_avg_position             7.1
ga4_sessions                 0.0
ga4_engaged_sessions         0.0
ctr                          0.0
engagement_rate              0.0
log_impressions              0.0
log_clicks                   0.0
log_sessions            0.693147
log_engaged_sessions         0.0
dtype: Float64


## 1. Build the Feature Vector

The feature vector is built from the core search and analytics performance metrics defined in the data contract.

The selected numeric features include GSC impressions, GSC clicks, average position, GA4 sessions, and GA4 engaged sessions. We also include engineered features such as CTR, engagement rate, and log-transformed traffic metrics.

Missing values are handled using median imputation for the current development sample. Missing-value indicators are added before imputation to preserve information about whether a value was originally unavailable.

The client and content IDs are excluded from the feature vector because they are identifiers used for grouping and joins, not predictive features. Report date is retained as context rather than used as a numeric model feature.

No future, label-derived, AI traffic, or product-decision fields are included.

In [11]:
import pandas as pd

feature_vector = feature_df.copy()

# Features that can contain missing values
impute_features = [
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate",
    "log_sessions",
    "log_engaged_sessions"
]

# Add missing indicators before imputation
for col in impute_features:
    feature_vector[f"{col}_missing"] = feature_vector[col].isna().astype(int)

# Median imputation for numeric features
for col in impute_features:
    feature_vector[col] = feature_vector[col].fillna(
        feature_vector[col].median()
    )

print("Remaining missing values:")
print(feature_vector[impute_features].isna().sum())

print("\nFeature vector shape:", feature_vector.shape)

feature_vector.head()

Remaining missing values:
gsc_avg_position        0
ga4_sessions            0
ga4_engaged_sessions    0
ctr                     0
engagement_rate         0
log_sessions            0
log_engaged_sessions    0
dtype: int64

Feature vector shape: (100000, 21)


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available,ga4_data_available,ctr,engagement_rate,...,log_clicks,log_sessions,log_engaged_sessions,gsc_avg_position_missing,ga4_sessions_missing,ga4_engaged_sessions_missing,ctr_missing,engagement_rate_missing,log_sessions_missing,log_engaged_sessions_missing
0,2026-03-01,20,0,3.350000,0,0,True,<NA>,0.000,0.0,...,0.000000,0.693147,0.0,0,1,1,0,1,1,1
1,2026-03-01,1,0,0.000000,0,0,True,<NA>,0.000,0.0,...,0.000000,0.693147,0.0,0,1,1,0,1,1,1
2,2026-03-01,125,1,4.928000,0,0,True,<NA>,0.008,0.0,...,0.693147,0.693147,0.0,0,1,1,0,1,1,1
3,2026-03-01,7,0,4.000000,0,0,True,<NA>,0.000,0.0,...,0.000000,0.693147,0.0,0,1,1,0,1,1,1
4,2026-03-01,11,0,2.272727,0,0,True,<NA>,0.000,0.0,...,0.000000,0.693147,0.0,0,1,1,0,1,1,1


### Feature Vector Result

After preprocessing, the feature vector contains 100,000 observations and 21 columns.

Missing values in the selected numeric features were imputed using the median calculated from the development sample. Missing indicators were added before imputation to preserve information about the original availability of each feature.

The final feature vector contains the selected performance metrics, engineered features, data-availability indicators, and missing-value indicators. Identifier fields such as client and content IDs are not included as model features.

All selected numeric features are now complete with no remaining missing values.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



The feature vector uses daily search and analytics performance measurements. The selected features are numeric; no categorical predictor is used.

| Feature | Meaning | Missing-value handling | Categorical? | Available-when? |
|---|---|---|---|---|
| `gsc_impressions` | Number of Google Search Console impressions for the content page on the report date. | No missing values observed in the development sample. | No | Available by the end of the report date. |
| `gsc_clicks` | Number of Google Search Console clicks for the content page on the report date. | No missing values observed in the development sample. | No | Available by the end of the report date. |
| `gsc_avg_position` | Average Google Search Console position for the content page. | Median imputation was used, with a missing indicator retained. Missingness corresponds to unavailable GSC data. | No | Available when GSC data is available for the report date. |
| `ga4_sessions` | Number of GA4 sessions associated with the content page. | Median imputation was used, with a missing indicator retained. Missingness is preserved separately rather than interpreted as zero. | No | Available when GA4 data is available for the report date. |
| `ga4_engaged_sessions` | Number of GA4 engaged sessions associated with the content page. | Median imputation was used, with a missing indicator retained. | No | Available when GA4 data is available for the report date. |
| `ctr` | Click-through rate calculated from GSC clicks and impressions. | Median imputation was used, with a missing indicator retained. | No | Available after the corresponding GSC daily measurements are available. |
| `engagement_rate` | Engaged sessions divided by GA4 sessions when GA4 sessions are greater than zero. | Median imputation was used, with a missing indicator retained. | No | Available when the corresponding GA4 measurements are available. |
| `log_impressions` | Log-transformed GSC impressions using `log(1 + impressions)`. | No missing values observed. | No | Available after the corresponding GSC measurement. |
| `log_clicks` | Log-transformed GSC clicks using `log(1 + clicks)`. | No missing values observed. | No | Available after the corresponding GSC measurement. |
| `log_sessions` | Log-transformed GA4 sessions using `log(1 + sessions)`. | Median imputation was used, with a missing indicator retained. | No | Available when GA4 data is available. |
| `log_engaged_sessions` | Log-transformed GA4 engaged sessions using `log(1 + engaged sessions)`. | Median imputation was used, with a missing indicator retained. | No | Available when GA4 data is available. |
| `*_missing` indicators | Binary indicators showing whether the corresponding value was originally missing. | Not imputed. Values are 0 or 1. | No | Available at feature construction time. |

The `report_date` is retained as context but is not used as a numeric model feature. Client and content identifiers are also excluded from the feature vector.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


I checked the feature set against three main leakage risks:

1. **Label-derived features:** fields that directly contain or encode the target or an outcome derived from it.
2. **Future or overlapping windows:** measurements that would only become available after the prediction moment.
3. **Product or decision-derived fields:** fields created from downstream decisions, scores, or product logic.

The current task is exploratory/clustering-based, so there is no predefined target label in this feature vector. The features are based on daily measurements and engineered transformations of those measurements.

I excluded identifiers, AI traffic fields, future-derived fields, and decision/product flags from the model feature set.

The following checks are used to verify that the selected feature columns do not contain known label-derived or excluded fields.

In [12]:
# Leakage and exclusion checks

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate",
    "log_impressions",
    "log_clicks",
    "log_sessions",
    "log_engaged_sessions",
    "gsc_avg_position_missing",
    "ga4_sessions_missing",
    "ga4_engaged_sessions_missing",
    "ctr_missing",
    "engagement_rate_missing",
    "log_sessions_missing",
    "log_engaged_sessions_missing"
]

forbidden_terms = [
    "trend",
    "label",
    "target",
    "ai_",
    "score",
    "decision",
    "product"
]

leaky_columns = [
    col for col in feature_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Number of model features:", len(feature_columns))
print("Potential leakage/excluded columns found:", leaky_columns)

assert len(leaky_columns) == 0, "Potential leakage or excluded field detected!"

print("Leakage name check passed.")

Number of model features: 18
Potential leakage/excluded columns found: []
Leakage name check passed.


### Future availability check

Each selected feature is derived from measurements associated with the same `report_date`. No future dates or future-window aggregates are used in the current feature construction.

The feature vector therefore uses information available for the content page at or by the end of the corresponding report date.

In [13]:
# Check the date range used in the development sample

print("Minimum report date:", feature_vector["report_date"].min())
print("Maximum report date:", feature_vector["report_date"].max())

print("\nNumber of unique report dates:",
      feature_vector["report_date"].nunique())

Minimum report date: 2026-03-01 00:00:00
Maximum report date: 2026-03-03 00:00:00

Number of unique report dates: 3


### Leakage test result

The development sample covers report dates from March 1 to March 3, 2026.

The selected features are calculated from measurements in the same daily performance record. No future-date aggregation, future-window feature, or target-derived feature is used in the current feature construction.

The leakage check also confirms that no known label-derived, AI traffic, product, decision, or identifier fields are included in the model feature list.

These checks support the feature design for the current development sample; they do not establish causal relationships or guarantee performance on unseen data.

In [14]:
# Final feature-set leakage check

excluded_patterns = [
    "trend",
    "label",
    "target",
    "ai_",
    "product",
    "decision",
    "score",
    "client_hash_id",
    "content_hash_id"
]

violations = []

for feature in feature_columns:
    for pattern in excluded_patterns:
        if pattern in feature.lower():
            violations.append((feature, pattern))

print("Model features checked:", len(feature_columns))
print("Potential violations:", violations)

assert len(violations) == 0, "Potential leakage/excluded feature detected."

print("Final leakage check passed.")

Model features checked: 18
Potential violations: []
Final leakage check passed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



The following fields were intentionally excluded from the feature vector:

- `client_hash_id` — identifier only; used for grouping or joins, not as a predictive feature.
- `content_hash_id` — identifier only; using it as a feature could allow the model to memorize specific content identities.
- `report_date` — retained as context rather than used as a numeric model feature.
- `ai_chatgpt` — AI traffic field excluded from the current feature design.
- `ai_perplexity` — AI traffic field excluded from the current feature design.
- `ai_gemini` — AI traffic field excluded from the current feature design.
- `ai_copilot` — AI traffic field excluded from the current feature design.
- `ai_claude` — AI traffic field excluded from the current feature design.
- `ai_meta` — AI traffic field excluded from the current feature design.
- `ai_other` — AI traffic field excluded from the current feature design.
- `trend_direction` — label-derived/outcome field; must not be used as a feature.
- `trend_pct` — label-derived/outcome field; must not be used as a feature.
- Future-window measurements — excluded because they would not be available at the prediction moment.
- Product flags or decision-derived scores — excluded because they are downstream of the decision process and can introduce leakage.


## Self-check

- [x] Every section above is filled with markdown reasoning and supporting code.
- [x] The feature vector was built on a manageable development sample.
- [x] Missing values were handled with median imputation and missing indicators.
- [x] Client and content identifiers were excluded from model features.
- [x] Leakage checks were performed and no potential leakage fields were detected.
- [x] The notebook runs top to bottom with no errors — to be verified with Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, measured, available, and decision-support.
- [x] Changes are committed to the repository.
- [x] The repository URL is submitted on the FlyRank assignment card.